In [2]:
from google.colab import files
uploaded = files.upload()

Saving orders.csv to orders (1).csv


In [3]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("/content/orders (1).csv")

In [7]:
print("RAW DATA OVERVIEW")
print(f"Total rows loaded: {len(df)}")
print(f"Missing values per column:\n{df.isnull().sum()}\n")

RAW DATA OVERVIEW
Total rows loaded: 25
Missing values per column:
order_id         0
customer_id      0
customer_name    0
region           0
order_date       1
delivery_date    2
amount           0
issue            9
dtype: int64



In [8]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["delivery_date"] = pd.to_datetime(df["delivery_date"], errors="coerce")

In [9]:
dropped_missing_order_date = df[df["order_date"].isna()]
df = df[df["order_date"].notna()].copy()

In [10]:
df["is_delivered"] = df["delivery_date"].notna()

In [11]:
df["issue"] = df["issue"].fillna("No issue reported")
df["issue"] = df["issue"].str.strip()
df.loc[df["issue"] == "", "issue"] = "No issue reported"

In [12]:
print(f"Rows dropped (missing order_date): {len(dropped_missing_order_date)}")
print(f"Rows still in transit: {(~df['is_delivered']).sum()}")
print(f"Rows remaining: {len(df)}")

Rows dropped (missing order_date): 1
Rows still in transit: 2
Rows remaining: 24


In [13]:
STANDARD_DELIVERY_DAYS = 5
today = pd.Timestamp("2026-06-17")

In [14]:
expected_date = df["order_date"] + pd.Timedelta(days=STANDARD_DELIVERY_DAYS)
reference_date = np.where(df["is_delivered"], df["delivery_date"], today)
reference_date = pd.to_datetime(reference_date)

delay_days = (reference_date - expected_date).dt.days
df["delay_days"] = np.where(delay_days > 0, delay_days, 0)
df["delayed"] = np.where(df["delay_days"] > 0, 1, 0)

df.head()

,order_id,customer_id,customer_name,region,order_date,delivery_date,amount,issue,is_delivered,delay_days,delayed
0,1,1,Asha Patel,West,2026-06-01,2026-06-08,1500.0,Courier delay,True,2,1
1,2,2,Ravi Kumar,North,2026-06-03,2026-06-07,800.0,No issue reported,True,0,0
2,3,3,Meena Iyer,South,2026-06-10,NaT,2200.0,Pending delivery,False,2,1
3,4,1,Asha Patel,West,2026-06-12,2026-06-19,950.0,Warehouse backlog,True,2,1
4,5,4,John Carter,East,2026-06-14,2026-06-25,3000.0,Address correction needed,True,6,1


In [15]:

delay_summary = (
    df.groupby(["customer_id", "customer_name"])["delayed"]
    .sum()
    .sort_values(ascending=False)
)
print("Top delayed customers (count):")
print(delay_summary.head(10))

Top delayed customers (count):
customer_id  customer_name 
1            Asha Patel        3
3            Meena Iyer        2
4            John Carter       2
6            Vikram Singh      1
10           Karan Shah        1
2            Ravi Kumar        0
5            Priya Nair        0
7            Sara Thomas       0
8            Arjun Mehta       0
9            Lena Fernandes    0
Name: delayed, dtype: int64


In [16]:
issues_only = df[df["issue"] != "No issue reported"]
issue_counts = issues_only["issue"].value_counts()
print("Most common issues:")
print(issue_counts)

Most common issues:
issue
Courier delay                     1
Pending delivery                  1
Warehouse backlog                 1
Address correction needed         1
Support unresponsive              1
Box damaged in transit            1
Minor routing delay               1
Tracking system issue             1
Wrong item sent                   1
Slight delay                      1
Repeat delay for this customer    1
Quality mismatch reported         1
Recurring delay pattern           1
Minor packaging issue             1
Courier delay again               1
Name: count, dtype: int64


In [17]:
df.to_csv("orders_cleaned.csv", index=False)
print("Saved orders_cleaned.csv")

Saved orders_cleaned.csv


In [18]:
from google.colab import files
files.download("orders_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>